In [1]:
from __future__ import annotations

import os                          # verificar rutas locales
import math                        # comprobar valores finitos
from dataclasses import dataclass  # estructura de la traza

import pandas as pd                # los datos del caso en tabla

print(f"pandas {pd.__version__} listo")

pandas 2.3.3 listo


In [2]:
RUTA_DATOS = "./resources/datos_muestra/"

In [3]:
COLUMNAS_REQUERIDAS = {
    "ventas.csv": {"fecha", "tienda_id", "producto_id"},
    "inventarios.csv": {"fecha", "tienda_id", "producto_id", "unidades_disponibles"},
    "tiendas.csv": {"tienda_id", "zona"},
    "productos.csv": {"producto_id", "rotacion_esperada", "precio_lista"},
}


def cargar_datos(ruta: str = RUTA_DATOS) -> dict[str, pd.DataFrame]:
    """Lee archivos locales y comprueba lo necesario para estos cálculos.

    No descarga, modifica ni sustituye datos. Acepta el nombre de carpeta
    'datos_muestra' que puede producir la descompresión del ZIP de Canvas.
    """
    if ruta == "muestra" and not os.path.isdir(ruta) and os.path.isdir("datos_muestra"):
        ruta = "datos_muestra"
    if not os.path.isdir(ruta):
        raise FileNotFoundError(
            f"No encuentro la carpeta {ruta!r}. Descarga 'Datos de muestra' de Canvas "
            "y descomprime sus CSV en una carpeta 'muestra' o 'datos_muestra'. "
            "En Jupyter, colócala junto al notebook; en Colab, súbela al panel de archivos.")
    tablas: dict[str, pd.DataFrame] = {}
    for archivo, columnas in COLUMNAS_REQUERIDAS.items():
        ubicacion = os.path.join(ruta, archivo)
        if not os.path.isfile(ubicacion):
            raise FileNotFoundError(f"Falta {ubicacion}. Revisa la carpeta descomprimida de Canvas.")
        marco = pd.read_csv(ubicacion)
        faltantes = columnas - set(marco.columns)
        if faltantes:
            raise ValueError(f"{archivo}: faltan columnas {sorted(faltantes)}. Revisa la descarga.")
        if marco.empty or marco[list(columnas)].isna().any().any():
            raise ValueError(f"{archivo}: tabla vacía o valores requeridos ausentes. Revisa la descarga.")
        for columna in columnas & {"tienda_id", "producto_id", "zona"}:
            if marco[columna].astype(str).str.strip().eq("").any():
                raise ValueError(f"{archivo}: hay valores vacíos en {columna}.")
        if "fecha" in columnas:
            marco["fecha"] = pd.to_datetime(marco["fecha"], errors="coerce")
            if marco["fecha"].isna().any():
                raise ValueError(f"{archivo}: hay fechas inválidas. Revisa los datos de origen.")
            marco["fecha"] = marco["fecha"].dt.normalize()
        for columna in columnas & {"unidades_disponibles", "rotacion_esperada", "precio_lista"}:
            marco[columna] = pd.to_numeric(marco[columna], errors="coerce")
            if not marco[columna].map(math.isfinite).all() or marco[columna].lt(0).any():
                raise ValueError(f"{archivo}: {columna} debe contener números finitos no negativos.")
        tablas[archivo.split(".")[0]] = marco
    for tabla, clave in (("tiendas", "tienda_id"), ("productos", "producto_id")):
        if tablas[tabla][clave].duplicated().any():
            raise ValueError(f"{tabla}.csv: {clave} debe ser único; revisa los duplicados.")
    for tabla in ("ventas", "inventarios"):
        for catalogo, clave in (("tiendas", "tienda_id"), ("productos", "producto_id")):
            if not tablas[tabla][clave].isin(tablas[catalogo][clave]).all():
                raise ValueError(f"{tabla}.csv: hay {clave} que no existen en {catalogo}.csv.")
    return tablas


datos = cargar_datos()
ventas = datos["ventas"]
inventarios = datos["inventarios"]
tiendas = datos["tiendas"]
productos = datos["productos"]
print(f"Entorno listo para este laboratorio. {len(ventas):,} filas de ventas, "
      f"{len(tiendas)} tiendas, {len(productos)} productos.")

Entorno listo para este laboratorio. 12,632 filas de ventas, 40 tiendas, 20 productos.


In [4]:
@dataclass(frozen=True)
class Paso:
    """Un paso de la traza del flujo: qué hizo, cómo y con qué resultado."""
    numero: int
    accion: str
    detalle: str
    resultado: str


class TrazaAgente:
    """Registro ordenado de los pasos programados del flujo, legible en vivo y en tabla."""

    def __init__(self) -> None:
        self.pasos: list[Paso] = []

    def registrar(self, accion: str, detalle: str, resultado: str) -> None:
        """Agrega un paso a la traza y lo muestra en el momento."""
        paso = Paso(len(self.pasos) + 1, accion, detalle, resultado)
        self.pasos.append(paso)
        print(f"[{paso.numero}] {paso.accion}\n    {paso.detalle}\n    -> {paso.resultado}\n")

    def como_tabla(self) -> pd.DataFrame:
        """Presenta la traza completa como tabla legible."""
        return pd.DataFrame(
            {"accion": [p.accion for p in self.pasos],
             "detalle": [p.detalle for p in self.pasos],
             "resultado": [p.resultado for p in self.pasos]},
            index=pd.RangeIndex(1, len(self.pasos) + 1, name="paso"))

In [5]:
# Flujo programado: cinco pasos fijos; no interpreta objetivos ni llama a una API.
traza = TrazaAgente()

# Ventana de siete fechas, anclada en la última fecha de ventas disponible.
fin = ventas.fecha.max()
semana = fin - pd.Timedelta(days=6)
inv_semana = inventarios[inventarios.fecha.between(semana, fin)]
traza.registrar(
    "PERCIBIR", f"Leo la ventana {semana.date()} a {fin.date()}, inclusive",
    f"{ventas.fecha.between(semana, fin).sum():,} filas de venta; "
    f"{len(inv_semana):,} de inventario")

# Una fecha se cuenta una sola vez aunque tenga varias observaciones en cero.
inv_cero = inv_semana[inv_semana.unidades_disponibles == 0]
dias_cero = inv_cero.groupby(["tienda_id", "producto_id"]).fecha.nunique()
traza.registrar(
    "DETECTAR", "Busco observaciones en cero; no prueban un quiebre continuo",
    f"{len(dias_cero)} combinaciones tienda-producto con al menos una fecha en cero; "
    f"{dias_cero.ge(2).sum()} con dos o más fechas distintas")

candidatos = (
    inv_cero
    .merge(productos[["producto_id", "rotacion_esperada", "precio_lista"]],
           on="producto_id", validate="many_to_one")
    .assign(costo_diario=lambda d: d.rotacion_esperada / 7 * d.precio_lista)
    .groupby(["tienda_id", "producto_id"])
    .agg(dias=("fecha", "nunique"), costo_diario=("costo_diario", "first"))
    .assign(costo_estimado=lambda d: d.dias * d.costo_diario)
    .sort_values("costo_estimado", ascending=False, kind="stable"))
caso = candidatos.head(1)

if caso.empty:
    traza.registrar("RAZONAR", "No hay candidatos para calcular un máximo",
                   "No se observaron inventarios en cero en la ventana disponible")
    traza.registrar("INVESTIGAR", "No hay un caso para comparar por producto y zona",
                   "La ausencia de candidatos no descarta problemas fuera de estos datos")
    traza.registrar("PROPONER", "No propongo una reposición",
                   "Revisar la cobertura y actualidad de los datos antes de concluir")
else:
    tienda_caso, producto_caso = caso.index[0]
    traza.registrar(
        "RAZONAR", "Aproximo costo: fechas en cero × rotación semanal / 7 × precio",
        f"Máximo estimado: {producto_caso} en {tienda_caso}, {int(caso.dias.iloc[0])} fechas, "
        f"${caso.costo_estimado.iloc[0]:,.0f}. No es venta perdida comprobada; "
        "usa rotación de referencia para tienda tamaño B, sin ajuste por tamaño")

    zonas = tiendas.set_index("tienda_id")["zona"]
    zona_caso = zonas.loc[tienda_caso]
    tiendas_zona = zonas[zonas == zona_caso].index
    mismas = candidatos[
        (candidatos.index.get_level_values("producto_id") == producto_caso)
        & candidatos.index.get_level_values("tienda_id").isin(tiendas_zona)]
    traza.registrar(
        "INVESTIGAR", "Cuento tiendas del mismo producto y de la misma zona",
        f"{len(mismas)} tiendas de la zona {zona_caso}, incluida la del caso, "
        "tienen observaciones en cero en la ventana. No demuestra simultaneidad "
        "ni una falla del centro de distribución")
    traza.registrar(
        "PROPONER", "Muestro una propuesta para revisión humana; no envío ni ejecuto órdenes",
        f"Revisar existencias y abastecimiento de {producto_caso} en {tienda_caso}; "
        "verificar la causa y la necesidad antes de autorizar una reposición")

print(f"Traza completa: {len(traza.pasos)} pasos programados. "
      "Solo se leyeron archivos y se mostraron resultados en este notebook.")
display(traza.como_tabla())

[1] PERCIBIR
    Leo la ventana 2026-06-22 a 2026-06-28, inclusive
    -> 3,057 filas de venta; 2,440 de inventario

[2] DETECTAR
    Busco observaciones en cero; no prueban un quiebre continuo
    -> 193 combinaciones tienda-producto con al menos una fecha en cero; 111 con dos o más fechas distintas

[3] RAZONAR
    Aproximo costo: fechas en cero × rotación semanal / 7 × precio
    -> Máximo estimado: P007 en T0004, 6 fechas, $1,723. No es venta perdida comprobada; usa rotación de referencia para tienda tamaño B, sin ajuste por tamaño

[4] INVESTIGAR
    Cuento tiendas del mismo producto y de la misma zona
    -> 2 tiendas de la zona Centro, incluida la del caso, tienen observaciones en cero en la ventana. No demuestra simultaneidad ni una falla del centro de distribución

[5] PROPONER
    Muestro una propuesta para revisión humana; no envío ni ejecuto órdenes
    -> Revisar existencias y abastecimiento de P007 en T0004; verificar la causa y la necesidad antes de autorizar una reposic

,accion,detalle,resultado
paso,,,
1,PERCIBIR,"Leo la ventana 2026-06-22 a 2026-06-28, inclusive","3,057 filas de venta; 2,440 de inventario"
2,DETECTAR,Busco observaciones en cero; no prueban un qui...,193 combinaciones tienda-producto con al menos...
3,RAZONAR,Aproximo costo: fechas en cero × rotación sema...,"Máximo estimado: P007 en T0004, 6 fechas, $1,7..."
4,INVESTIGAR,Cuento tiendas del mismo producto y de la mism...,"2 tiendas de la zona Centro, incluida la del c..."
5,PROPONER,Muestro una propuesta para revisión humana; no...,Revisar existencias y abastecimiento de P007 e...


In [ ]:
# DECISIÓN | Tus tres respuestas, entre las comillas
respuesta_1 = """
Los pasos programados en el flujo son PERCIBIR, DETECTAR, RAZONAR, INVESTIGAR y PROPONER. El sistema ejecuta una secuencia lineal de código en la que el orden y las decisiones de cálculo están definidos de antemano de forma rígida por el programador. Para afirmar que el sistema "elige su ruta" (comportamiento agéntico), se necesitaría evidencia de que un modelo de IA evalúa dinámicamente la situación, interpreta el objetivo de negocio en lenguaje natural, decide en tiempo de ejecución qué herramientas o funciones invocar y adapta sus pasos o estrategia según los resultados intermedios.

"""
respuesta_2 = """
•	Información que puede consultar: Archivos CSV locales con datos del negocio (ventas.csv, inventarios.csv, tiendas.csv y productos.csv).
•	Acciones que puede realizar: Cargar y validar dataframes en pandas, realizar filtros, agregaciones numéricas para estimar costos aproximados y desplegar un resumen en texto dentro del notebook.
•	Acción que requeriría autorización humana: Cualquier intervención en el mundo real o en sistemas transaccionales, tales como autorizar una orden de reposición, modificar el inventario en el ERP, emitir una compra o enviar alertas automáticas al gerente de zona.

"""
respuesta_3 = """
•	Conclusión aceptable: La identificación de fechas específicas en las que se registraron observaciones de inventario disponible en cero para ciertas combinaciones de tienda y producto dentro del rango analizado (dato duro verificado en los CSV).
•	Conclusión a verificar: La hipótesis de que este inventario en cero constituye un quiebre de anaquel con "venta perdida real", la suposición de que se trata de una falla regional del centro de distribución y la estimación del costo.
•	Justificación: El código únicamente detecta registros en cero y multiplica las fechas afectadas por un promedio de rotación semanal de referencia (tienda tamaño B) y el precio de lista. No considera el tamaño real de la tienda, la demanda verdadera del periodo, la existencia de stock en bodega ni si el desabasto fue consecutivo o parcial.
"""

def validar_respuestas(respuestas: list[str]) -> list[int]:
    """Comprueba presencia de texto; no evalúa la calidad de los argumentos."""
    return [n for n, r in enumerate(respuestas, start=1)
            if not isinstance(r, str) or not r.strip()]


pendientes = validar_respuestas([respuesta_1, respuesta_2, respuesta_3])
for n in (1, 2, 3):
    estado = "PENDIENTE" if n in pendientes else "TEXTO REGISTRADO (sin evaluar)"
    print(f"Respuesta {n}: {estado}")
if pendientes:
    print("Completa las respuestas pendientes antes de preparar y subir tu PDF a Canvas.")
else:
    print("Hay texto en las tres respuestas. Revisa tus argumentos y sube el PDF con tus respuestas y evidencia de ejecución.")

Respuesta 1: TEXTO REGISTRADO (sin evaluar)
Respuesta 2: TEXTO REGISTRADO (sin evaluar)
Respuesta 3: TEXTO REGISTRADO (sin evaluar)
Hay texto en las tres respuestas. Revisa tus argumentos y sube el PDF con tus respuestas y evidencia de ejecución.


In [7]:
def pruebas_del_notebook() -> None:
    """Cuatro grupos de comprobaciones técnicas, no una evaluación académica."""
    # 1. Se cargaron las tablas y las columnas requeridas.
    for nombre, columnas in COLUMNAS_REQUERIDAS.items():
        tabla = datos[nombre.split(".")[0]]
        assert columnas <= set(tabla.columns), f"{nombre}: faltan columnas"
        assert not tabla.empty, f"{nombre}: llegó vacío"

    # 2. La secuencia es fija y explícita, incluso si no hay candidatos.
    assert [p.accion for p in traza.pasos] == [
        "PERCIBIR", "DETECTAR", "RAZONAR", "INVESTIGAR", "PROPONER"]

    # 3. Máximo, días distintos y comparación dentro de la zona.
    assert caso.empty == candidatos.empty
    if not caso.empty:
        assert caso.costo_estimado.iloc[0] == candidatos.costo_estimado.max()
        assert candidatos.dias.between(1, 7).all()
        assert int(caso.dias.iloc[0]) == int(dias_cero.loc[caso.index[0]])
        assert mismas.index.get_level_values("tienda_id").isin(tiendas_zona).all()

    # 4. Detectar texto no equivale a comprobar una respuesta correcta.
    assert validar_respuestas(["", " ", "x"]) == [1, 2]
    assert validar_respuestas([None, 12, "texto"]) == [1, 2]
    assert validar_respuestas(["a", "b", "c"]) == []
    print("4/4 grupos de comprobaciones técnicas en verde. "
          "No validan tus argumentos, la causa del quiebre ni una decisión de reposición.")


pruebas_del_notebook()

4/4 grupos de comprobaciones técnicas en verde. No validan tus argumentos, la causa del quiebre ni una decisión de reposición.
